In [ ]:
!pip install transformers datasets scikit-learn torch
import math
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
fake = pd.read_csv("Fake (2).csv", engine='python', on_bad_lines='skip')
true = pd.read_csv("True (2).csv", engine='python', on_bad_lines='skip')

fake['label'] = 0
true['label'] = 1

data = pd.concat([fake, true], ignore_index=True)
data = data[['text', 'label']]
data = data.sample(10000).reset_index(drop=True)

print(f"Dataset shape: {data.shape}")
print(f"Label distribution:\n{data['label'].value_counts()}")
data.head()

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'], data['label'], test_size=0.2, random_state=42
)
print(f"Train size: {len(train_texts)}, Test size: {len(test_texts)}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LENGTH = 256

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

train_encodings = tokenize(train_texts)
test_encodings  = tokenize(test_texts)

print("Tokenization complete.")
print(f"Sample token IDs (first 10): {train_encodings['input_ids'][0][:10]}")

In [ ]:
from transformers import BertModel
_tmp = BertModel.from_pretrained('bert-base-uncased')
pe_shape = _tmp.embeddings.position_embeddings.weight.shape
print(f"BERT Learned Positional Embedding table shape: {pe_shape}")
print(f"  → {pe_shape[0]} positions × {pe_shape[1]} hidden dimensions")
print(f"  → Trainable parameters: {pe_shape[0] * pe_shape[1]:,}")
print("Positional encoding is active and will run automatically inside BertEmbeddings.")
del _tmp

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids':      train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels':         list(train_labels)
})

test_dataset = Dataset.from_dict({
    'input_ids':      test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels':         list(test_labels)
})

print(f"Train dataset: {train_dataset}")
print(f"Test  dataset: {test_dataset}")

In [ ]:
# Teacher Model
teacher_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2,attn_implementation="eager"
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_dir='./logs',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

trainer_teacher = Trainer(
    model=teacher_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_teacher.train()

In [ ]:
print("Teacher Model Evaluation:")
teacher_results = trainer_teacher.evaluate()
print(teacher_results)

In [ ]:
# Student Model — weights copied from the fine-tuned teacher (first 6 layers)
# Per the paper: "compressed 6-layer version which keeps the initial six
# transformer blocks of the teacher."
# Copying teacher weights (not loading fresh bert-base-uncased) gives the
# student task-specific knowledge right from the start.

import copy

student_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2, attn_implementation="eager"
)

# Copy fine-tuned teacher encoder weights (layers 0–5) into the student
with torch.no_grad():
    # Encoder layers
    for layer_idx in range(6):
        student_layer = student_model.bert.encoder.layer[layer_idx]
        teacher_layer = teacher_model.bert.encoder.layer[layer_idx]
        student_layer.load_state_dict(copy.deepcopy(teacher_layer.state_dict()))

    # Embeddings (token, position, layer-norm) — also from the fine-tuned teacher
    student_model.bert.embeddings.load_state_dict(
        copy.deepcopy(teacher_model.bert.embeddings.state_dict())
    )

# Truncate to 6 layers
student_model.bert.encoder.layer = student_model.bert.encoder.layer[:6]

print(f"Teacher layers : 12")
print(f"Student layers : {len(student_model.bert.encoder.layer)}")
print("Student initialised from fine-tuned teacher weights (layers 0-5 + embeddings).")


In [ ]:
class DistillationTrainer(Trainer):

    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher     = teacher_model
        self.temperature = temperature
        self.alpha       = alpha

        self.teacher.eval()
        for param in self.teacher.parameters():
            param.requires_grad = False

    @staticmethod
    def _adaptive_k(teacher_probs):
        """
        Compute per-sample adaptive Top-K based on teacher prediction entropy.
        High-confidence (low entropy) → K=1 (sharp distribution).
        Uncertain (high entropy)      → higher K to preserve richer signal.
        K is clipped to [1, num_classes].
        """
        num_classes = teacher_probs.size(1)
        # Shannon entropy: H = -sum(p * log(p + eps))
        entropy = -(teacher_probs * (teacher_probs + 1e-9).log()).sum(dim=1)  # (B,)
        max_entropy = math.log(num_classes + 1e-9)
        # Normalised entropy in [0, 1]; map to K in [1, num_classes]
        k_float = 1 + entropy / (max_entropy + 1e-9) * (num_classes - 1)
        k = k_float.round().long().clamp(1, num_classes)
        return k  # (B,)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs_student = model(**inputs)
        student_logits  = outputs_student.logits

        with torch.no_grad():
            outputs_teacher = self.teacher(**inputs)
            teacher_logits  = outputs_teacher.logits

        teacher_probs = F.softmax(teacher_logits, dim=1)   # (B, C)

        # ── Adaptive Top-K sparse mask ─────────────────────────────────────
        k_per_sample = self._adaptive_k(teacher_probs)     # (B,)

        batch_size, num_classes = teacher_probs.shape
        sparse_mask = torch.zeros_like(teacher_probs)      # (B, C)

        for b in range(batch_size):
            k_b = k_per_sample[b].item()
            topk_vals, topk_idx = torch.topk(teacher_probs[b], k=k_b)
            sparse_mask[b, topk_idx] = 1.0

        # ── Importance weighting: scale kept entries by their probability ──
        # More probable classes carry proportionally higher weight in the
        # sparse distribution, so informative classes have greater influence.
        sparse_weighted = teacher_probs * sparse_mask      # zero out non-top-K
        row_sums = sparse_weighted.sum(dim=1, keepdim=True).clamp(min=1e-9)
        sparse_teacher = sparse_weighted / row_sums        # renormalise to sum=1

        # ── Loss terms ────────────────────────────────────────────────────
        # L_sparse: KL divergence against importance-weighted sparse teacher
        loss_sparse = F.kl_div(
            F.log_softmax(student_logits, dim=1),
            sparse_teacher,
            reduction='batchmean'
        )

        # L_CE: standard cross-entropy against hard ground-truth labels
        loss_hard = F.cross_entropy(student_logits, labels)

        # Paper equation: L = alpha * L_sparse + (1 - alpha) * L_CE
        loss = self.alpha * loss_sparse + (1.0 - self.alpha) * loss_hard

        return (loss, outputs_student) if return_outputs else loss


trainer_student = DistillationTrainer(
    model=student_model,
    teacher_model=teacher_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_student.train()


In [ ]:
print("GREEN-LM (Sparse Knowledge Distillation) Evaluation:")
student_results = trainer_student.evaluate()
print(student_results)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EnergyAdaptiveBERT(nn.Module):

    def __init__(self, student_model, exit_threshold=0.90):
        super().__init__()

        self.exit_threshold = exit_threshold
        self.config = student_model.config

        hidden_size = student_model.config.hidden_size
        num_labels = student_model.config.num_labels
        num_layers = len(student_model.bert.encoder.layer)

        self.embeddings = student_model.bert.embeddings
        self.all_layers = student_model.bert.encoder.layer

        self.exit_classifiers = nn.ModuleList([
            nn.Linear(hidden_size, num_labels) for _ in range(num_layers)
        ])

        self.last_exit_layer = None

    def forward(self, input_ids, attention_mask):
        hidden = self.embeddings(input_ids=input_ids)

        attn_mask_4d = attention_mask[:, None, None, :].float()
        attn_mask_4d = (1.0 - attn_mask_4d) * -10000.0

        confidence = None

        for i, layer in enumerate(self.all_layers):

            self_attn_out = layer.attention(hidden, attention_mask=attn_mask_4d)
            hidden = self_attn_out[0]

            intermediate = layer.intermediate(hidden)
            hidden = layer.output(intermediate, hidden)

            if i < 2:
                continue


            cls_repr = hidden[:, 0, :]

            logits = self.exit_classifiers[i](cls_repr)
            probs = F.softmax(logits, dim=1)


            top2 = torch.topk(probs, 2, dim=1).values
            confidence = (top2[:, 0] - top2[:, 1])

            # Early exit condition
            if confidence.max().item() >= self.exit_threshold:
                self.last_exit_layer = i + 1
                return logits, i + 1, confidence.max().item()

        # Final layer (no early exit)
        cls_repr = hidden[:, 0, :]
        logits = self.exit_classifiers[len(self.all_layers) - 1](cls_repr)

        self.last_exit_layer = len(self.all_layers)

        return logits, len(self.all_layers), confidence.max().item()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

energy_model = EnergyAdaptiveBERT(student_model, exit_threshold=0.90)
energy_model.to(device)
energy_model.eval()

print(f"EnergyAdaptiveBERT built on: {device}")
print(f"Exit threshold: {energy_model.exit_threshold}")

In [ ]:
from torch.optim import Adam
import torch.nn.functional as F
import random

print("Training exit classifiers (multi-exit training)...")
print()

for param in energy_model.parameters():
    param.requires_grad = False

for clf in energy_model.exit_classifiers:
    for param in clf.parameters():
        param.requires_grad = True

optimizer = Adam(energy_model.exit_classifiers.parameters(), lr=1e-3)
energy_model.train()

NUM_EPOCHS = 2
batch_size = 32

for epoch in range(NUM_EPOCHS):
    total_loss = 0
    total_correct = 0
    total_samples = 0

    indices = list(range(len(train_dataset)))
    random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        batch = train_dataset[batch_idx]

        input_ids = torch.tensor(batch['input_ids']).to(device)
        attention_mask = torch.tensor(batch['attention_mask']).to(device)
        labels = torch.tensor(batch['labels']).to(device)

        hidden = energy_model.embeddings(input_ids=input_ids)

        attn_mask_4d = attention_mask[:, None, None, :].float()
        attn_mask_4d = (1.0 - attn_mask_4d) * -10000.0

        total_batch_loss = 0

        for i, layer in enumerate(energy_model.all_layers):

            self_attn_out = layer.attention(hidden, attention_mask=attn_mask_4d)
            hidden = self_attn_out[0]

            intermediate = layer.intermediate(hidden)
            hidden = layer.output(intermediate, hidden)

            if i < 2:
                continue

            # Train EVERY exit classifier
            cls_repr = hidden[:, 0, :]
            logits = energy_model.exit_classifiers[i](cls_repr)

            loss = F.cross_entropy(logits, labels)
            total_batch_loss += loss

        # Backprop using combined loss
        optimizer.zero_grad()
        total_batch_loss.backward()
        optimizer.step()

        total_loss += total_batch_loss.item() * len(batch_idx)

        # Accuracy from final exit
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += len(batch_idx)

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {avg_loss:.4f} — Acc: {acc:.4f}")

for param in energy_model.parameters():
    param.requires_grad = True

energy_model.eval()

print("\nMulti-exit training complete.")

In [ ]:
import requests
import torch
import torch.nn.functional as F
from datetime import datetime

def get_carbon_intensity(zone="IN-SO", api_key=None):
    """
    Simulates South India grid carbon intensity based on time of day.
    Based on POSOCO/CEA data for IN-SO grid:
      - Peak morning/evening (coal fills demand gap): ~750 gCO2/kWh
      - Midday solar peak (Tamil Nadu + Karnataka farms): ~500 gCO2/kWh
      - Night baseload (coal dominant): ~650 gCO2/kWh
    """
    # hour = datetime.now().hour
    import pytz

    ist = pytz.timezone('Asia/Kolkata')
    hour = datetime.now(ist).hour
    if 6 <= hour <= 10:
        carbon = 750    # Morning peak — solar ramping, coal filling gap
    elif 11 <= hour <= 16:
        carbon = 500    # Midday — solar at peak, cleanest grid window
    elif 17 <= hour <= 22:
        carbon = 780    # Evening peak — solar gone, coal/gas rush in
    else:
        carbon = 650    # Night — low demand, coal baseload
    print(f"[Carbon Monitor] Hour={hour}h | Zone=IN-SO | Carbon={carbon} gCO2/kWh | ", end="")
    if carbon <= 550:
        print("Grid Status: 🟢 CLEAN")
    elif carbon <= 700:
        print("Grid Status: 🟡 MODERATE")
    else:
        print("Grid Status: 🔴 DIRTY")
    return carbon


def carbon_aware_predict(teacher_model, energy_model, inputs,
                         base_threshold=0.95, zone="IN-SO", api_key=None):
    carbon = get_carbon_intensity(zone, api_key)
    # Clean grid  → lower threshold → student handles more (energy use is guilt-free)\n",
    # Dirty grid  → higher threshold → student must be very confident or teacher steps in
    # When carbon intensity is HIGH  → lower threshold → student handles more
    #   (routing to small model reduces energy on a dirty grid)
    # When carbon intensity is LOW   → higher threshold → student must be very
    #   confident, otherwise teacher (accuracy-first on a clean grid)
    # This implements the paper's design: threshold decreases as carbon rises.
    threshold = base_threshold - (carbon - 400) / 1000
    threshold = max(threshold, 0.50)   
    threshold = min(threshold, 0.99)   
    teacher_model.eval()
    energy_model.eval()

    with torch.no_grad():
        logits, exit_layer, confidence = energy_model(**inputs)
        pred = torch.argmax(F.softmax(logits, dim=1), dim=1)

        if confidence >= threshold:
            return {
                "prediction":       pred.item(),
                "confidence":       round(confidence, 6),
                "model_used":       "STUDENT",
                "exit_layer":       exit_layer,
                "carbon_path":      "LOW CARBON",
                "carbon_intensity": carbon,
                "threshold_used":   round(threshold, 4)
            }

        teacher_out = teacher_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        t_probs = F.softmax(teacher_out.logits, dim=1)
        t_conf, t_pred = torch.max(t_probs, dim=1)

        return {
            "prediction":       t_pred.item(),
            "confidence":       round(t_conf.item(), 6),
            "model_used":       "TEACHER FALLBACK",
            "exit_layer":       12,
            "carbon_path":      "HIGH CARBON",
            "carbon_intensity": carbon,
            "threshold_used":   round(threshold, 4)
        }


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Evaluating on full test set with Carbon-Aware Output Layer

all_preds        = []
all_labels       = []
student_count    = 0
teacher_count    = 0
early_exit_count = 0

routing_log = []

for i in range(len(test_dataset)):
    sample     = test_dataset[i]
    true_label = sample['labels']

    inputs = {
        'input_ids': torch.tensor(sample['input_ids']).unsqueeze(0).to(device),
        'attention_mask': torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)
    }

    result = carbon_aware_predict(
        teacher_model=teacher_model,
        energy_model=energy_model,
        inputs=inputs,
        base_threshold=0.95,
        zone="IN-SO",
        api_key=None  # Using time-based simulation
    )

    all_preds.append(result['prediction'])
    all_labels.append(true_label)

    # Determine energy mode manually
    if result['model_used'] == 'STUDENT':
        energy_mode = f"EXIT @ LAYER {result['exit_layer']}"
    else:
        energy_mode = "TEACHER (12 LAYERS)"

    # Logging
    routing_log.append({
        'sample':      i + 1,
        'true_label':  label_map[true_label],
        'predicted':   label_map[result['prediction']],
        'correct':     '✓' if result['prediction'] == true_label else '✗',
        'model_used':  result['model_used'],
        'exit_layer':  result['exit_layer'],
        'energy_mode': energy_mode,
        'carbon_path': result['carbon_path'],
        'confidence':  result['confidence']
    })

    # Counts
    if result['model_used'] == 'STUDENT':
        student_count += 1

        # Early exit check
        if result['exit_layer'] < 6:
            early_exit_count += 1
    else:
        teacher_count += 1

    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(test_dataset)} samples...")

total = len(test_dataset)
acc   = accuracy_score(all_labels, all_preds)

# ── Summary ─────────────────────────────────────────
print("\n" + "=" * 60)
print("FULL TEST SET RESULTS — CARBON-AWARE OUTPUT LAYER")
print("=" * 60)

print(f"Total Test Samples    : {total}")
print(f"Overall Accuracy      : {acc:.4f} ({acc*100:.2f}%)")

print(f"Student Used          : {student_count}/{total} ({student_count/total*100:.1f}%) — LOW CARBON")

print(f"  └─ Early Exit       : {early_exit_count}/{student_count} "
      f"({early_exit_count/max(student_count,1)*100:.1f}% of student calls)")

print(f"Teacher Fallback Used : {teacher_count}/{total} ({teacher_count/total*100:.1f}%) — HIGH CARBON")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['FAKE', 'REAL']))

# ── Routing table ───────────────────────────────────
routing_df = pd.DataFrame(routing_log)

print("\n" + "=" * 60)
print("PER-SAMPLE ROUTING LOG (first 30 samples)")
print("=" * 60)
print(routing_df.head(30).to_string(index=False))

# ── Energy breakdown ────────────────────────────────
print("\n" + "=" * 60)
print("ENERGY MODE BREAKDOWN")
print("=" * 60)
print(routing_df['energy_mode'].value_counts().to_string())

# ── Save CSV ────────────────────────────────────────
routing_df.to_csv("routing_log.csv", index=False)
print("\nFull routing log saved to routing_log.csv")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating TEACHER model...")

teacher_model.eval()

teacher_preds = []
teacher_labels = []

for i in range(len(test_dataset)):
    sample = test_dataset[i]
    true_label = sample['labels']

    inputs = {
        'input_ids': torch.tensor(sample['input_ids']).unsqueeze(0).to(device),
        'attention_mask': torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)
    }

    with torch.no_grad():
        outputs = teacher_model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    teacher_preds.append(pred)
    teacher_labels.append(true_label)

teacher_acc = accuracy_score(teacher_labels, teacher_preds)
acc = accuracy_score(all_labels, all_preds)
print("\nTEACHER RESULTS")
print("=" * 40)
print(f"Accuracy: {teacher_acc:.4f} ({teacher_acc*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(teacher_labels, teacher_preds, target_names=['FAKE','REAL']))
print("\n" + "=" * 50)
print("MODEL COMPARISON")
print("=" * 50)

print(f"Teacher Accuracy  : {teacher_acc:.4f} ({teacher_acc*100:.2f}%)")
print(f"GREEN-LM Accuracy : {acc:.4f} ({acc*100:.2f}%)")

drop = teacher_acc - acc

print(f"\nAccuracy Drop     : {drop:.4f} ({drop*100:.2f}%)")

if drop < 0.02:
    print("Very small drop — excellent tradeoff")
elif drop < 0.05:
    print("Acceptable drop — good efficiency gain")
else:
    print("High drop — may need tuning")

In [ ]:
import time
import torch

num_samples = 2000
teacher_total = 0
student_total = 0
teacher_times = []
student_times = []

# 🔥 Warm-up (important)
for i in range(50):
    sample = test_dataset[i]
    inputs = {
        'input_ids': torch.tensor(sample['input_ids']).unsqueeze(0).to(device),
        'attention_mask': torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)
    }
    with torch.no_grad():
        _ = teacher_model(**inputs)
        _ = energy_model(**inputs)

# 🚀 Actual timing
for i in range(num_samples):
    sample = test_dataset[i]
    inputs = {
        'input_ids': torch.tensor(sample['input_ids']).unsqueeze(0).to(device),
        'attention_mask': torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)
    }

    # Teacher timing
    torch.cuda.synchronize() if device.type == "cuda" else None
    start = time.perf_counter()

    with torch.no_grad():
        _ = teacher_model(**inputs)

    torch.cuda.synchronize() if device.type == "cuda" else None
    elapsed = time.perf_counter() - start

    teacher_total += elapsed
    teacher_times.append(elapsed)

    # Student timing
    torch.cuda.synchronize() if device.type == "cuda" else None
    start = time.perf_counter()

    with torch.no_grad():
        _ = energy_model(**inputs)

    torch.cuda.synchronize() if device.type == "cuda" else None
    elapsed = time.perf_counter() - start

    student_total += elapsed
    student_times.append(elapsed)

# ✅ Averages
teacher_avg = teacher_total / num_samples
student_avg = student_total / num_samples
speedup = teacher_avg / student_avg

print("\n===== AVERAGE INFERENCE TIME =====")
print(f"Teacher  Model Avg Time : {teacher_avg:.6f} sec")
print(f"GREEN-LM Model Avg Time : {student_avg:.6f} sec")
print(f"Speedup                 : {speedup:.2f}x")

# 🔥 Extra (recommended)
print(f"Teacher Std Dev : {torch.tensor(teacher_times).std():.6f}")
print(f"Student Std Dev : {torch.tensor(student_times).std():.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

samples = list(range(1, num_samples + 1))

# ✅ FIX: Use cumulative average (correct for per-sample timings)
teacher_avg_curve = np.cumsum(teacher_times) / np.arange(1, num_samples + 1)
student_avg_curve = np.cumsum(student_times) / np.arange(1, num_samples + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ===============================
# Plot 1: Inference time comparison
# ===============================
axes[0].plot(samples, teacher_avg_curve, label="Teacher (12-layer BERT)", color='#e74c3c')
axes[0].plot(samples, student_avg_curve, label="GREEN-LM (Energy Adaptive)", color='#2ecc71')

axes[0].set_title("Inference Latency (Running Average)")
axes[0].set_xlabel("Number of Samples")
axes[0].set_ylabel("Time per Sample (seconds)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)


axes[0].text(0.6, 0.8, f"{speedup:.2f}x speedup",
             transform=axes[0].transAxes,
             fontsize=11,
             bbox=dict(facecolor='white', alpha=0.7))

# ===============================
# Plot 2: Carbon routing breakdown
# ===============================
labels_pie = ['Student\n(LOW CARBON)', 'Teacher Fallback\n(HIGH CARBON)']
sizes_pie = [student_count, teacher_count]
colors_pie = ['#2ecc71', '#e74c3c']

axes[1].pie(
    sizes_pie,
    labels=labels_pie,
    colors=colors_pie,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)

axes[1].set_title("Carbon-Aware Routing Distribution")

# ===============================
# Final layout & save
# ===============================
plt.tight_layout()
plt.savefig("green_lm_results.png", dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved as green_lm_results.png")